# Input Tensor (`x`)

Given:

```python
B,T,C = 4,8,32
x = torch.randn(B,T,C)
```

---

## What do `B`, `T`, and `C` represent?

| Symbol | Meaning |
|----------|----------|
| B | Batch Size (number of sequences processed together) |
| T | Sequence Length (number of tokens) |
| C | Embedding Dimension (features per token) |

Example:

```python
(4,8,32)
```

means:

```text
4 sequences
8 tokens per sequence
32 features per token
```

---

## Why is `x.shape = (B,T,C)`?

Each token needs an embedding.

```text
Batch
 └── Tokens
      └── Features
```

Therefore:

```python
(B,T,C)
```

Example:

```text
Batch 0
[
 token0 -> [32 numbers]
 token1 -> [32 numbers]
 ...
 token7 -> [32 numbers]
]
```

---

## Why is `C` called embedding dimension?

Each token is represented by a vector:

```python
token = [x₁,x₂,x₃,...,x₃₂]
```

The number of values in that vector is the embedding dimension.

```python
C = len(token_embedding)
```

Example:

```python
[0.2, -1.1, 0.7]
```

↓

```python
C = 3
```

---

## Why is `T` sometimes called the time dimension?

Historically sequence models processed data step-by-step:

```text
t=0
t=1
t=2
...
```

For text:

```text
"I love transformers"
```

becomes:

```text
Token0 -> I
Token1 -> love
Token2 -> transformers
```

So:

```text
Time ≈ Position in sequence
```

Not necessarily actual clock time.

---

## What does `x[b,t]` contain?

Indexing removes one dimension:

```python
x.shape
=
(B,T,C)
```

```python
x[b,t]
```

Shape:

```python
(C,)
```

Contains:

```text
Embedding vector of token t
from batch b
```

Example:

```python
x[0,3]
```

↓

```python
[
 0.12,
 -0.44,
 1.23,
 ...
]
```

Length:

```python
C
```

---

## What does `x[b,t,c]` represent?

```python
x[b,t,c]
```

Shape:

```python
scalar
```

Meaning:

```text
Feature c
of token t
inside batch b
```

Example:

```python
x[0,3,5]
```

↓

```python
0.817
```

Interpretation:

```text
Batch 0
Token 3
Feature 5
```

---

## Shape Breakdown

| Expression | Shape | Meaning |
|------------|---------|---------|
| `x` | `(B,T,C)` | Entire tensor |
| `x[b]` | `(T,C)` | One sequence |
| `x[b,t]` | `(C,)` | One token embedding |
| `x[b,t,c]` | `()` | Single feature value |

---

## Mental Model

```text
x 
(B,T,C)
↓
Batch
↓
Tokens
↓
Embedding Features
```

Example:

```text
x[0]

[
 token0 -> [32 features]
 token1 -> [32 features]
 ...
]
```

Each token is represented by a vector.

Each vector has `C` learned features.

## `torch.randn`

Given:

```python
B,T,C = 4,8,32
x = torch.randn(B,T,C)
```

Shape:

```python
(4,8,32)
```

---

### What does `torch.randn(B,T,C)` generate?

Generates:

```python
B*T*C
```

independent random numbers sampled from a normal distribution.

Example:

```python
torch.randn(2,3)
```

↓

```python
[
 [ 0.12, -0.81, 1.34],
 [-1.02,  0.57, 0.21]
]
```

---

### How can a normal distribution exist in 3 dimensions?

It doesn't (here).

PyTorch is **not** generating a 3D Gaussian.

It simply generates:

```python
4 * 8 * 32
=
1024
```

independent random numbers and arranges them into:

```python
(4,8,32)
```

Think:

```text
Generate numbers first
↓
Arrange into tensor later
```

---

### Is it generating a 3D Gaussian?

❌ No

```python
torch.randn(4,8,32)
```

is equivalent to:

```python
Generate 1024 random values
↓
Reshape to (4,8,32)
```

Each element is sampled independently.

---

### Why random values?

Neural networks learn by adjusting weights.

If everything starts with:

```python
0
```

then every neuron behaves identically.

Example:

```text
Neuron 1 → same output
Neuron 2 → same output
Neuron 3 → same output
```

Gradients become identical.

Learning breaks.

---

### Why specifically a Normal Distribution?

Most values are small:


::contentReference[oaicite:0]{index=0}


Typical values:

```text
≈ -3 to +3
```

Properties:

| Property | Benefit |
|----------|----------|
| Mean ≈ 0 | No directional bias |
| Small values | Stable training |
| Positive & Negative values | Break symmetry |
| Random | Different neurons learn different things |

---

### Shape Breakdown

| Expression | Meaning |
|------------|---------|
| `torch.randn(32)` | 32 random values |
| `torch.randn(8,32)` | 8 vectors of length 32 |
| `torch.randn(4,8,32)` | 4 batches × 8 tokens × 32 features |

---

### Mental Model

```python
torch.randn(4,8,32)
```

↓

```text
Generate 1024 random numbers
↓
Arrange into

(B,T,C)

=
(4,8,32)
```

The tensor shape does **not** affect the distribution.

It only affects how the numbers are organized.

# Linear Layers (`nn.Linear`)

## What is `nn.Linear(in_features, out_features)`?

A learnable linear transformation:

```python
y = xWᵀ + b
```

Transforms:

```text
(*, in_features)
        ↓
(*, out_features)
```

Example:

```python
nn.Linear(32,16)
```

↓

```text
32 features → 16 features
```

---

## Is `nn.Linear` a function, tensor, matrix, or object?

| Type | Answer |
|--------|--------|
| Function | ❌ |
| Tensor | ❌ |
| Matrix | ❌ |
| Object | ✅ |

Contains:

```python
weight
bias (optional)
```

and knows how to apply them.

---

## Does it contain parameters internally?

✅ Yes

```python
layer = nn.Linear(32,16)

layer.weight
layer.bias
```

These are the learnable parameters.

---

## Where are parameters stored?

Inside the layer object:

```python
layer.weight
```

Shape:

```python
(16,32)
```

```python
layer.bias
```

Shape:

```python
(16,)
```

---

## How are weights initialized?

Automatically by PyTorch.

```python
layer = nn.Linear(32,16)
```

creates random weights.

Example:

```python
[
 [ 0.03, -0.12, ...],
 [-0.08,  0.07, ...],
 ...
]
```

Purpose:

```text
Break symmetry
Allow learning
```

---

## What does `bias=False` mean?

Without bias:

```python
y = xWᵀ
```

Only matrix multiplication.

```python
nn.Linear(32,16,bias=False)
```

---

## What changes when `bias=True`?

With bias:

```python
y = xWᵀ + b
```

Bias:

```python
b.shape = (out_features,)
```

Example:

```python
xWᵀ = [4,5]

b = [1,2]
```

↓

```python
[5,7]
```

---

## Shape Example

Input:

```python
x.shape = (B,T,32)
```

Layer:

```python
nn.Linear(32,16)
```

Operation:

```text
(B,T,32)
@
(32,16)
=
(B,T,16)
```

Output:

```python
(B,T,16)
```

Only the last dimension changes.

---

## Why only the last dimension?

Documentation:

```text
(*, Hin)
```

means:

```text
Anything before Hin is preserved
Only Hin is transformed
```

Examples:

| Input | Output |
|---------|---------|
| `(32)` | `(16)` |
| `(8,32)` | `(8,16)` |
| `(4,8,32)` | `(4,8,16)` |
| `(100,4,8,32)` | `(100,4,8,16)` |

---

## Mental Model

```text
nn.Linear

=
Learnable Matrix Multiplication
```

```text
Input Features
      ↓
Weighted Combination
      ↓
Output Features
```

Each output feature sees **all input features**.

---

## Quick Reference

| Question | Answer |
|----------|---------|
| What is it? | Learnable linear transform |
| Stores parameters? | Yes |
| Parameters? | Weight + Bias |
| Weight shape | `(out_features, in_features)` |
| Bias shape | `(out_features,)` |
| Initialized by | PyTorch |
| `bias=False` | No bias term |
| `bias=True` | Adds bias |
| Changes which dimension? | Last dimension only |

### Core Formula

```python
nn.Linear(Hin, Hout)
```

```text
(*, Hin)
    ↓
(*, Hout)
```

```text
(B,T,Hin)
@
(Hin,Hout)
=
(B,T,Hout)
```

# Linear Layers: Shape Transformations

## Why does `nn.Linear(32,16)` convert `(B,T,32)` → `(B,T,16)`?

Given:

```python
nn.Linear(32,16)
```

PyTorch learns:

```text
(32×16)
```

worth of weights.

For each token:

```text
(1×32)
@
(32×16)
=
(1×16)
```

Therefore:

```text
(B,T,32)
@
(32,16)
=
(B,T,16)
```

Only the feature dimension changes.

---

## Why does PyTorch only care about the last dimension?

Because `nn.Linear` is defined as:

```text
(*, Hin)
```

where:

```text
Hin = in_features
```

PyTorch interprets:

```python
(4,8,32)
```

as:

```text
(*,32)
```

Everything before `32` is treated as batch-like dimensions.

Only the final dimension must match:

```python
in_features=32
```

---

## What does `(*, Hin)` mean?

Documentation:

```text
Input: (*, Hin)
Output: (*, Hout)
```

Meaning:

```text
* = any number of dimensions
Hin = last dimension
```

Examples:

| Input | Output |
|---------|---------|
| `(32)` | `(16)` |
| `(8,32)` | `(8,16)` |
| `(4,8,32)` | `(4,8,16)` |
| `(100,4,8,32)` | `(100,4,8,16)` |

Everything before `Hin` is preserved.

---

## Why can `nn.Linear` accept 3D tensors?

Because it operates independently on every vector whose length is:

```python
Hin
```

Example:

```python
x.shape = (4,8,32)
```

Think:

```text
32-d vector
32-d vector
32-d vector
...
```

There are:

```python
4 * 8 = 32
```

such vectors.

Each one gets transformed:

```text
(1×32)
@
(32×16)
=
(1×16)
```

Result:

```python
(4,8,16)
```

---

## Where does `(1×32)` come from?

A single token:

```python
x[b,t]
```

Shape:

```python
(32,)
```

Mathematically we view it as:

```text
(1×32)
```

a row vector.

This makes matrix multiplication easier to visualize:

```text
(1×32)
@
(32×16)
=
(1×16)
```

The leading `1` simply means:

```text
one token embedding
```

---

## Visual Example

```python
x.shape = (2,3,4)
```

```text
Batch 0
[
 token0 -> [4 nums]
 token1 -> [4 nums]
 token2 -> [4 nums]
]

Batch 1
[
 token0 -> [4 nums]
 token1 -> [4 nums]
 token2 -> [4 nums]
]
```

Apply:

```python
nn.Linear(4,2)
```

Each token:

```text
(1×4)
@
(4×2)
=
(1×2)
```

Result:

```python
(2,3,2)
```

---

## Mental Model

```text
nn.Linear does NOT care about:

B
T
extra dimensions

It only cares about:

Hin
```

```text
(*, Hin)
      ↓
Linear
      ↓
(*, Hout)
```

Everything before the last dimension is preserved.

# Linear Layers: Mathematics & Learning

## What operation is actually performed inside `nn.Linear`?

Without bias:

```python
y = xWᵀ
```

With bias:

```python
y = xWᵀ + b
```

Core operation:

```text
Matrix Multiplication
```

---

## Is it just matrix multiplication?

✅ Yes

```python
nn.Linear(Hin,Hout)
```

is essentially:

```text
(Hin) @ (Hin×Hout)
=
(Hout)
```

plus optional bias.

---

## How does a linear layer combine features?

Example:

```python
x = [1,2,3]
```

```python
W =
[
 [1,0],
 [0,1],
 [1,1]
]
```

```text
(1×3)
@
(3×2)
=
(1×2)
```

Output:

```python
[
 1*1 + 2*0 + 3*1,
 1*0 + 2*1 + 3*1
]
```

↓

```python
[4,5]
```

Each output is a weighted combination of all inputs.

---

## How does every output dimension see all input dimensions?

Weight matrix:

```python
(3×2)
```

```text
[
 [1,0],
 [0,1],
 [1,1]
]
```

| Output | Uses Inputs |
|----------|----------|
| Output 1 | x₁, x₂, x₃ |
| Output 2 | x₁, x₂, x₃ |

Every column of `W` creates one output feature.

---

## Why is this called a projection?

Input:

```python
[1,2,3]
```

Output:

```python
[4,5]
```

Same information source.

Different coordinate system.

```text
Embedding
     ↓
Linear Transform
     ↓
Projected Embedding
```

The vector is being represented in a new space.

---

## Why not simply call it a new embedding?

Because:

```text
New embedding
```

doesn't explain where it came from.

```text
Projection
```

implies:

```text
Original vector
      ↓
Linear transformation
      ↓
New representation
```

The relationship is preserved.

---

# Learning

## How does a linear layer learn which features matter?

Initially:

```python
W = random
```

Nothing meaningful exists.

During training:

```text
Forward Pass
     ↓
Loss
     ↓
Backpropagation
     ↓
Update Weights
```

Repeated millions of times.

---

## If weights start random, how do meaningful features emerge?

Initially:

```text
Feature 7 weight = random
Feature 12 weight = random
Feature 25 weight = random
```

After training:

```text
Useful features
↑ larger weights

Less useful features
↑ smaller weights
```

The optimizer discovers which combinations reduce loss.

---

## Is this entirely due to backpropagation?

Almost.

Learning pipeline:

```text
Random Initialization
        ↓
Forward Pass
        ↓
Loss
        ↓
Backpropagation
        ↓
Gradient Descent
        ↓
Updated Weights
```

Backprop computes:

```text
How should weights change?
```

Gradient descent applies the change.

---

## What if `out_features < in_features`?

Example:

```python
nn.Linear(32,16)
```

```text
32 → 16
```

Compression.

```text
Keep important information
Discard redundant information
```

Shape:

```text
(B,T,32)
@
(32,16)
=
(B,T,16)
```

---

## What if `out_features = in_features`?

Example:

```python
nn.Linear(32,32)
```

```text
32 → 32
```

No compression.

No expansion.

Still useful because:

```text
Old Features
      ↓
Mixed Together
      ↓
New Features
```

Shape:

```text
(B,T,32)
@
(32,32)
=
(B,T,32)
```

Representation changes.

Dimension stays the same.

---

## What if `out_features > in_features`?

Example:

```python
nn.Linear(32,64)
```

```text
32 → 64
```

Expansion.

Creates more feature channels.

Shape:

```text
(B,T,32)
@
(32,64)
=
(B,T,64)
```

Common in Transformer MLPs:

```text
d_model
   ↓
4*d_model
   ↓
d_model
```

---

## Mental Model

```text
Linear Layer

=
Learn useful feature combinations
```

```text
Input Features
      ↓
Weighted Mixing
      ↓
New Features
```

Not:

```text
Feature 1 → Feature 1
Feature 2 → Feature 2
```

Instead:

```text
All Features
      ↓
Mixed Together
      ↓
New Representation
```

---

## Quick Reference

| Case | Meaning |
|--------|--------|
| `32 → 16` | Compression |
| `32 → 32` | Re-mapping |
| `32 → 64` | Expansion |
| Random weights | Initialization |
| Backprop | Computes updates |
| Gradient Descent | Applies updates |
| Projection | Same information, new space |
| Linear Layer | Learn feature combinations |

# Design Questions

## Why use `Linear` instead of `BiLinear`?

Goal of Q/K projection:

```text
Embedding
      ↓
Learn a new representation space
```

Linear already does exactly that:

```python
q = xWq
k = xWk
```

BiLinear needs two inputs:

```python
y = x₁ᵀAx₂
```

Attention hasn't started comparing tokens yet.

We first need to transform each token independently.

---

## Why is a linear transformation sufficient here?

A linear layer can:

```text
Rotate
Scale
Stretch
Compress
Mix Features
```

which is enough to create a new representation space.

Example:

```text
32 features
     ↓
Linear
     ↓
16 learned features
```

The actual non-linearity comes later from:

```python
softmax()
```

and other layers in the network.

---

## What properties of linear transformations make them useful?

| Property | Benefit |
|-----------|-----------|
| Learnable | Can adapt during training |
| Efficient | Fast matrix multiplication |
| Differentiable | Easy to optimize |
| Mixes features | Creates richer representations |
| Dimension change | Compress or expand features |

---

# Keys and Queries

## Why are `key` and `query` different if both are:

```python
nn.Linear(32,16)
```

Because:

```python
key   = nn.Linear(32,16)
query = nn.Linear(32,16)
```

creates:

```python
Wk
```

and

```python
Wq
```

separately.

Each layer gets its own random weights.

---

## Why isn't:

```python
key(x) == query(x)
```

Because:

```python
Wk ≠ Wq
```

Example:

```text
Wk =
[
 ...
]
```

```text
Wq =
[
 ...
]
```

Different weights:

```text
Same Input
     ↓
Different Transformations
     ↓
Different Outputs
```

---

## How are `Wk` and `Wq` initialized?

Automatically by PyTorch.

```python
key   = nn.Linear(...)
query = nn.Linear(...)
```

Each layer samples its own random weights.

Therefore:

```text
Wk ≠ Wq
```

from the very beginning.

---

## Why are they different from the start?

Random initialization.

If both were identical:

```text
Wk = Wq
```

then:

```text
Key Space
=
Query Space
```

and the model would lose flexibility.

Random initialization breaks symmetry.

---

# Conceptual

## How does one projection become a "query"?

Initially:

```text
Query = random projection
```

Nothing in the code says:

```text
"This is searching."
```

During training:

```text
Predict next token
      ↓
Backpropagation
      ↓
Update Wq
```

The model discovers:

```text
Some features are useful for asking:
"What information do I need?"
```

Those become query features.

---

## How does another projection become a "key"?

Initially:

```text
Key = random projection
```

Training discovers:

```text
Some features are useful for saying:
"What information do I contain?"
```

Those become key features.

---

## Who decides which is which?

Nobody explicitly.

Not:

```python
query = SearchingLayer()
```

Not:

```python
key = InformationLayer()
```

They start as:

```text
Random Projection A
Random Projection B
```

The names:

```text
Query
Key
```

are interpretations of what training causes them to do.

---

## Is there anything in the code enforcing this?

❌ No semantic enforcement.

Only:

```python
q = xWq
k = xWk
```

The special behavior emerges because of:

```python
q @ k.T
```

and the training objective.

---

## How does training make them specialize?

Training repeatedly does:

```text
Forward Pass
      ↓
Loss
      ↓
Backpropagation
      ↓
Update Wq and Wk
```

Eventually:

```text
Wq learns:
"What am I looking for?"

Wk learns:
"What information do I contain?"
```

because that minimizes loss.

---

## Mental Model

Initially:

```text
Embedding
      ↓
Random Projection A
      ↓
q
```

```text
Embedding
      ↓
Random Projection B
      ↓
k
```

After training:

```text
q
=
Features useful for searching
```

```text
k
=
Features useful for matching
```

The model invents these roles.

They are not hardcoded.

---

## Quick Reference

| Question | Answer |
|-----------|-----------|
| Why Linear? | Learn a new representation space |
| Why not BiLinear? | Only one input exists at this stage |
| Why different Q/K? | Different weight matrices |
| Why `key(x) ≠ query(x)`? | `Wk ≠ Wq` |
| Who decides Query/Key meaning? | Training |
| Hardcoded roles? | No |
| How do they specialize? | Backpropagation + loss minimization |
| Initial state? | Random projections |

# Representation Learning

## What does "projection of an embedding" actually mean?

Original embedding:

```python
x = [1,2,3]
```

Apply:

```python
q = xWq
```

```text
(1×3)
@
(3×2)
=
(1×2)
```

Result:

```python
q = [4,5]
```

Same token.

Different coordinates.

```text
Embedding
     ↓
Linear Transform
     ↓
Projected Embedding
```

---

## What is a representation space?

A space where vectors live.

Example:

```python
[1,2,3]
```

lives in:

```text
R³
```

After:

```python
xWq
```

↓

```python
[4,5]
```

it lives in:

```text
R²
```

Different space.

Different coordinates.

Different notion of similarity.

---

## Why learn a new representation space?

Raw embeddings contain:

```text
Grammar
Syntax
Meaning
Position
Etc.
```

Not all information is useful for attention.

Learned projections can focus on:

```text
Features useful for matching
```

Example:

```text
Raw Embedding
      ↓
Projection
      ↓
Only useful matching signals remain
```

---

## Why not work directly in embedding space?

Possible:

```python
x @ x.T
```

```text
(T×C)
@
(C×T)
=
(T×T)
```

But then:

```text
Similarity is forced to use
the raw embedding space.
```

No learning.

No specialization.

With:

```python
q = xWq
k = xWk
```

the model first learns:

```text
What features should matter?
```

and then computes similarity.

---

## Mental Model

```text
Embedding Space
       ↓
Learn Projection
       ↓
Query/Key Space
       ↓
Measure Similarity
```

---

# Matrix Multiplication

## Why does:

```python
(B,T,16) @ (B,16,T)
```

work?

Ignore batch first:

```text
(T×16)
@
(16×T)
=
(T×T)
```

Rule:

```text
(m×n)
@
(n×o)
=
(m×o)
```

The inner dimensions:

```text
16 and 16
```

match.

Result:

```text
(T×T)
```

Batch dimension is preserved:

```text
(B,T,T)
```

---

## Why doesn't:

```python
(B,T,16) @ (B,T,16)
```

work?

Ignoring batch:

```text
(T×16)
@
(T×16)
```

Rule requires:

```text
(m×n)
@
(n×o)
```

but we get:

```text
(T×16)
@
(T×16)
```

Inner dimensions:

```text
16 and T
```

do not match.

Therefore:

```text
Invalid
```

---

## How does batch matrix multiplication work?

PyTorch treats:

```python
(B,T,16)
```

as:

```text
B separate matrices
```

Example:

```python
(4,8,16)
```

means:

```text
Matrix 0 -> (8×16)
Matrix 1 -> (8×16)
Matrix 2 -> (8×16)
Matrix 3 -> (8×16)
```

and

```python
(4,16,8)
```

means:

```text
Matrix 0 -> (16×8)
Matrix 1 -> (16×8)
Matrix 2 -> (16×8)
Matrix 3 -> (16×8)
```

PyTorch computes:

```text
(8×16) @ (16×8) = (8×8)
```

for each batch independently.

Result:

```python
(4,8,8)
```

---

## Batch MatMul Example

```python
A.shape = (2,3,4)
B.shape = (2,4,5)
```

Per batch:

```text
(3×4)
@
(4×5)
=
(3×5)
```

Therefore:

```python
(2,3,5)
```

---

## Why does `q @ k.T` produce `(T,T)`?

```python
q.shape = (B,T,16)
k.shape = (B,T,16)
```

Transpose:

```python
k.T
```

↓

```python
(B,16,T)
```

Then:

```text
(T×16)
@
(16×T)
=
(T×T)
```

Each cell:

```text
q(token_i)
·
k(token_j)
```

So:

```text
Rows    = querying token
Columns = candidate token
```

Result:

```text
Similarity Matrix
```

between every pair of tokens.

---

## Quick Reference

| Expression | Result |
|------------|---------|
| `(m×n) @ (n×o)` | `(m×o)` |
| `(T×16) @ (16×T)` | `(T×T)` |
| `(T×16) @ (T×16)` | ❌ Invalid |
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| `(B,T,16) @ (B,16,T)` | `(B,T,T)` |

### Mental Model

```text
q = What am I looking for?
k = What information do I contain?

q @ k.T

=
How relevant is token j
to token i?
```

# Dot Product

## What is the dot product actually computing?

For two vectors:

```python
a = [a₁,a₂,...]
b = [b₁,b₂,...]
```

Dot product:

$$
a \cdot b
=
\sum_i a_i b_i
$$

Example:

```python
[1,2,3]
·
[4,5,6]
```

```text
=
1×4 + 2×5 + 3×6

=
32
```

---

## Why does it measure similarity?

Each feature votes:

```text
Same sign  → positive contribution
Opposite sign → negative contribution
Near zero → little contribution
```

Example:

| Vectors | Dot Product | Interpretation |
|----------|----------|----------|
| `[1,2] · [1,2]` | `5` | Similar |
| `[1,2] · [-1,-2]` | `-5` | Opposite |
| `[1,0] · [0,1]` | `0` | Unrelated |

---

## What does a large dot product mean?

Large positive:

```text
Features agree
```

Example:

```python
[3,4]
·
[2,5]
=
26
```

Many dimensions point in the same direction.

---

## What does a negative dot product mean?

Example:

```python
[1,2]
·
[-1,-2]
=
-5
```

Features disagree.

The vectors point in opposite directions.

---

## Mental Model

```text
Dot Product

=
Agreement Score
```

Large:

```text
"We match"
```

Small/Negative:

```text
"We don't match"
```

---

# Attention

## What exactly is:

```python
q @ k.T
```

measuring?

Shapes:

```text
(T×16)
@
(16×T)
=
(T×T)
```

Each cell:

```text
q(token_i)
·
k(token_j)
```

Example:

```text
[
 q₀·k₀  q₀·k₁  q₀·k₂
 q₁·k₀  q₁·k₁  q₁·k₂
 q₂·k₀  q₂·k₁  q₂·k₂
]
```

Every token compared with every token.

---

## Is it importance?

❌ Not directly.

A token can be important but irrelevant to the current token.

Dot product does NOT ask:

```text
"How important are you?"
```

---

## Is it similarity?

✅ Yes.

It measures:

```text
How similar is my query
to your key?
```

---

## Is it relevance?

✅ More accurate than "importance".

The attention score asks:

```text
How useful are you
for what I'm currently looking for?
```

---

## Similarity vs Importance

| Concept | Meaning |
|----------|----------|
| Importance | Important overall |
| Similarity | Matches my features |
| Relevance | Useful for my current query |

Attention mainly learns:

```text
Relevance
```

---

# Representation Space

## If `q` and `k` live in a learned space, what does similarity mean?

Not:

```text
Raw embedding similarity
```

Instead:

```text
Similarity after learning
which features matter.
```

---

## What kinds of features are being compared?

The model decides.

Examples:

```text
Subject information
Location information
Gender information
Verb agreement
Entity references
```

No human explicitly defines them.

Training discovers useful features.

---

## What is the dot product measuring after projection?

Before projection:

```python
x @ x.T
```

measures:

```text
Similarity in embedding space
```

After projection:

```python
q = xWq
k = xWk
```

```python
q @ k.T
```

measures:

```text
Similarity in a learned space
```

where:

```text
Wq and Wk learned
which features should matter.
```

---

## Full Pipeline

```text
Embedding
      ↓
q = xWq
k = xWk
      ↓
Learn useful features
      ↓
q @ k.T
      ↓
Similarity Scores
      ↓
Softmax
      ↓
Attention Weights
```

---

## The Deepest Interpretation

```python
q @ k.T
```

does NOT ask:

```text
Who is important?
```

It asks:

```text
Given what token i is looking for,
which token j best matches it?
```

That matching score is the dot product.

---

## Quick Reference

| Expression | Meaning |
|------------|----------|
| `a · b` | Feature agreement |
| Large positive | Strong match |
| Negative | Opposite features |
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| Attention score | Relevance score |
| Softmax output | Attention weight |

### Mental Model

```text
Query:
"What am I looking for?"

Key:
"What information do I contain?"

Dot Product:
"How well do we match?"
```

# Matrix Multiplication Shape Rules

## Why does

```text
(m×n) @ (n×o)
=
(m×o)
```

work?

Because matrix multiplication computes:

```text
Row from A
      ·
Column from B
```

for every output cell.

The inner dimensions:

```text
n and n
```

must match so the dot product can happen.

---

## What are the matching dimensions?

```text
(m×n)
@
(n×o)
```

Matching dimensions:

```text
n and n
```

Think:

```text
Columns of A
=
Rows of B
```

If they don't match:

```text
Multiplication is invalid
```

---

## What are the surviving dimensions?

```text
(m×n)
@
(n×o)
=
(m×o)
```

Survive:

```text
m and o
```

Disappear:

```text
n
```

---

## Quick Examples

### Example 1

```text
(2×3)
@
(3×4)
=
(2×4)
```

Matching:

```text
3 and 3
```

Surviving:

```text
2 and 4
```

---

### Example 2

```text
(T×16)
@
(16×T)
=
(T×T)
```

Matching:

```text
16 and 16
```

Surviving:

```text
T and T
```

---

### Example 3

```text
(T×16)
@
(T×16)
```

❌ Invalid

Matching dimensions should be:

```text
16 and T
```

which usually don't match.

---

## Mental Model

```text
Inside dimensions match

Outside dimensions survive
```

```text
(m×n)
@
(n×o)
=
(m×o)
```

---

# Transpose

## Why do we need:

```python
k.transpose(-2,-1)
```

Before:

```python
q.shape = (B,T,16)
k.shape = (B,T,16)
```

We want:

```python
q @ k.T
```

Using shape rules:

```text
(T×16)
@
(16×T)
=
(T×T)
```

So we must convert:

```text
(T×16)
```

into:

```text
(16×T)
```

which is exactly what transpose does.

---

## What does transpose actually do?

It swaps dimensions.

Example:

```python
(8,16)
```

↓

```python
transpose
```

↓

```python
(16,8)
```

Rows become columns.

Columns become rows.

---

## Why swap only the last two dimensions?

Original:

```python
(B,T,16)
```

We want:

```python
(B,16,T)
```

Only the matrix dimensions participate in:

```python
@
```

The batch dimension is not part of the matrix multiplication itself.

So we only swap:

```text
T ↔ 16
```

---

## Why use negative indices instead of positive ones?

For:

```python
(B,T,16)
```

Positive indices:

```text
0 → B
1 → T
2 → 16
```

Negative indices:

```text
-3 → B
-2 → T
-1 → 16
```

Therefore:

```python
transpose(-2,-1)
```

means:

```text
swap T and 16
```

---

### Why is this preferred?

Works regardless of tensor rank.

Example:

```python
(T,16)
```

```python
(B,T,16)
```

```python
(H,B,T,16)
```

In all cases:

```python
transpose(-2,-1)
```

still means:

```text
swap the last two dimensions
```

---

## Why is the batch dimension untouched?

Given:

```python
(B,T,16)
@
(B,16,T)
```

PyTorch performs:

```text
Batch 0:
(T×16) @ (16×T)

Batch 1:
(T×16) @ (16×T)

Batch 2:
(T×16) @ (16×T)
```

independently.

The batch dimension is simply:

```text
How many matrices do we have?
```

not part of the matrix itself.

---

## Visual

Before:

```text
(B,T,16)
```

```text
Batch
 └── Matrix(T×16)
```

After:

```text
(B,16,T)
```

```text
Batch
 └── Matrix(16×T)
```

Batch remains exactly where it is.

Only the matrix is transposed.

---

## Quick Reference

| Expression | Result |
|------------|---------|
| `(m×n) @ (n×o)` | `(m×o)` |
| Matching dimensions | Inner dimensions |
| Surviving dimensions | Outer dimensions |
| `transpose(-2,-1)` | Swap last two dimensions |
| `(B,T,16)` | Original |
| `(B,16,T)` | After transpose |
| Batch dimension | Preserved |
| Why transpose? | Make inner dimensions match |

### Mental Model

```text
Matrix Multiplication:
Match the inside
Keep the outside

Transpose:
Rotate the matrix
without touching the batch dimension
```

# Alternatives

## Why use dot product instead of cosine similarity?

Cosine similarity:

$$
\cos(\theta)
=
\frac{a \cdot b}{||a||\,||b||}
$$

Measures:

```text
Direction only
```

Ignores magnitude.

---

### Example

```python
a = [1,1]
b = [10,10]
```

Cosine:

```text
1.0
```

because they point in the same direction.

Dot Product:

```text
20
```

because both direction and magnitude matter.

---

## Why does attention prefer dot products?

| Reason | Benefit |
|----------|----------|
| Faster | No norm computation |
| Simple | Just matrix multiplication |
| Differentiable | Easy optimization |
| Magnitude matters | Strong signals can be emphasized |
| GPU friendly | Extremely efficient |

---

## Then why scale by √d?

Large dimensions:

```python
q·k
```

can become very large.

Large values:

```python
softmax(...)
```

↓

```text
Very peaky probabilities
```

Small gradients.

Therefore:

```python
(q @ k.T) / √d
```

keeps values stable.

---

## Why use projected embeddings instead of raw embeddings?

Raw embeddings:

```python
x @ x.T
```

```text
(T×C)
@
(C×T)
=
(T×T)
```

Measures similarity in the original embedding space.

---

Projected embeddings:

```python
q = xWq
k = xWk
```

```python
q @ k.T
```

Measures similarity in a learned space.

The model first learns:

```text
Which features matter?
```

then computes similarity.

---

## Mental Model

```text
Raw Embeddings
      ↓
Generic Similarity

Projected Embeddings
      ↓
Task-Specific Similarity
```

---

# Attention Scores (`wei`)

## Why is:

```python
wei.shape = (B,T,T)
```

Given:

```python
q.shape = (B,T,16)
k.shape = (B,T,16)
```

Transpose:

```python
k.T
```

↓

```python
(B,16,T)
```

Multiply:

```text
(B,T,16)
@
(B,16,T)
=
(B,T,T)
```

because:

```text
(T×16)
@
(16×T)
=
(T×T)
```

---

## Why do we get a token-token matrix?

Every token compares itself against every token.

Example:

```text
Token0 vs Token0
Token0 vs Token1
Token0 vs Token2

Token1 vs Token0
Token1 vs Token1
Token1 vs Token2

...
```

Number of comparisons:

```text
T × T
```

Therefore:

```python
(T,T)
```

---

## What does each cell contain?

Cell:

```python
wei[i,j]
```

contains:

```text
q(token_i)
·
k(token_j)
```

which is:

```text
Similarity / Relevance Score
```

before softmax.

---

## Visual Example

Suppose:

```text
T = 3
```

Then:

```python
wei
=
[
 [q₀·k₀, q₀·k₁, q₀·k₂],
 [q₁·k₀, q₁·k₁, q₁·k₂],
 [q₂·k₀, q₂·k₁, q₂·k₂]
]
```

Shape:

```python
(3,3)
```

---

## What does:

```python
wei[i,j]
```

mean?

It answers:

```text
How relevant is token j
to token i?
```

or

```text
How much should token i
pay attention to token j?
```

Notice the direction:

```text
Row = Querying token
Column = Candidate token
```

---

### Example

Sentence:

```text
"The cat sat"
```

Suppose:

```python
wei[2,1]
```

Then:

```text
Row 2 → "sat"
Column 1 → "cat"
```

Meaning:

```text
How relevant is "cat"
to "sat"?
```

before softmax normalization.

---

## Before vs After Softmax

Before:

```python
wei
```

```text
Raw similarity scores
```

Example:

```python
[
 [2,5,1]
]
```

After:

```python
softmax(wei)
```

```python
[
 [0.05,0.91,0.04]
]
```

Now:

```text
Attention weights
```

that sum to:

```text
1
```

---

## Quick Reference

| Expression | Meaning |
|------------|----------|
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| `wei.shape` | `(B,T,T)` |
| Row `i` | Query token |
| Column `j` | Candidate token |
| `wei[i,j]` | Relevance of token `j` to token `i` |
| Before Softmax | Scores |
| After Softmax | Attention Weights |

### Mental Model

```text
Query:
"What am I looking for?"

Key:
"What information do I contain?"

wei[i,j]:
"How well does token j
match what token i needs?"
```

# Raw Embeddings vs Q/K

## Why not simply do:

```python
x @ x.T
```

Shapes:

```text
(T×C)
@
(C×T)
=
(T×T)
```

This would measure:

```text
Similarity in the raw embedding space
```

The problem:

```text
The model cannot choose
which features matter.
```

It is forced to use whatever information happens to be stored in the embedding.

---

## Why introduce `Wq` and `Wk`?

Instead:

```python
q = xWq
k = xWk
```

```text
(T×C)
@
(C×16)
=
(T×16)
```

Now attention becomes:

```python
q @ k.T
```

```text
(T×16)
@
(16×T)
=
(T×T)
```

The model first learns:

```text
Which features should be compared?
```

and only then computes similarity.

---

## Raw Embeddings vs Learned Projections

| Raw Embeddings | Q/K Projections |
|---------------|---------------|
| Fixed similarity | Learned similarity |
| Uses all features equally | Learns useful features |
| No specialization | Query/Key specialization |
| `x @ x.T` | `q @ k.T` |
| Generic matching | Task-specific matching |

---

# Learning

## What additional flexibility do learned projections provide?

Without projections:

```text
Token
    ↓
Embedding
    ↓
Similarity
```

With projections:

```text
Token
    ↓
Embedding
    ↓
Learn Features
    ↓
Similarity
```

The extra step allows the model to decide:

```text
What should matter?
```

instead of us deciding.

---

## How do projections amplify useful features?

Suppose embedding:

```python
[
 subject,
 tense,
 color,
 position
]
```

Maybe attention only needs:

```text
subject
position
```

Training may learn:

```text
Large weights
for useful features

Small weights
for irrelevant features
```

Example:

```python
[2,3,1,4]
```

↓

```python
[10,12]
```

Useful information becomes stronger.

---

## How does this happen mathematically?

Suppose:

```python
x = [2,5,1]
```

and:

```python
W =
[
 [5],
 [4],
 [0.1]
]
```

Then:

```text
(1×3)
@
(3×1)
=
(1×1)
```

```python
2×5 + 5×4 + 1×0.1
=
30.1
```

Feature 1 and 2 dominate.

Feature 3 barely contributes.

The projection has amplified important signals.

---

## Is attention a kind of feature amplification?

✅ Partly yes.

Q and K layers learn:

```text
Which features deserve attention?
```

The projection can strengthen:

```text
important features
```

and weaken:

```text
unimportant features
```

before similarity is computed.

---

## Is attention a kind of feature extraction?

✅ Also yes.

This is the subtle part.

The projection creates:

```text
New Features
```

not merely stronger old ones.

Example:

```text
Old Features:
gender
tense
position
```

↓

```text
New Learned Feature:
"is this token likely
the subject of a sentence?"
```

That feature did not explicitly exist before.

---

## Amplification vs Extraction

| Amplification | Extraction |
|---------------|-------------|
| Increase useful signals | Create new signals |
| Larger weights | New feature combinations |
| Keep existing information | Transform information |
| "Make feature stronger" | "Invent better feature" |

Attention does both.

---

## The Deepest Interpretation

```python
q = xWq
k = xWk
```

is NOT:

```text
Find similarity
```

It is:

```text
Learn a space where
similarity becomes meaningful
```

Only after that do we compute:

```python
q @ k.T
```

---

## Mental Model

```text
Raw Embedding
      ↓
Contains lots of information
      ↓
Projection (Wq/Wk)
      ↓
Extract + Amplify useful features
      ↓
Learned Representation Space
      ↓
Dot Product
      ↓
Relevance Scores
```

---

## Quick Reference

| Question | Answer |
|-----------|-----------|
| Why not `x @ x.T`? | No learned similarity |
| Why Q/K? | Learn what matters before comparing |
| What do projections add? | Flexibility |
| What gets amplified? | Useful features |
| What gets suppressed? | Irrelevant features |
| Is attention feature amplification? | Partly yes |
| Is attention feature extraction? | Also yes |
| Best description | Similarity in a learned feature space |

### One-Line Summary

```text
x @ x.T

=
"Are these embeddings similar?"

q @ k.T

=
"Are these embeddings similar according to features the model learned are important?"
```

# Softmax

## What is the exact softmax formula?

For a row:

```python
[x₁,x₂,x₃,...]
```

Softmax:

$$
softmax(x_i)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

Example:

```python
[1,2,3]
```

↓

Exponentiate:

```python
[e¹,e²,e³]
```

↓

```python
[2.7,7.4,20.1]
```

↓

Normalize:

```python
[
 2.7/30.2,
 7.4/30.2,
 20.1/30.2
]
```

↓

```python
[0.09,0.24,0.67]
```

---

## Why exponentiate first?

Exponentiation amplifies differences.

Example:

```python
[1,2,3]
```

Difference:

```text
1
```

After exponentiation:

```python
[2.7,7.4,20.1]
```

Difference becomes much larger.

This helps attention strongly prefer better matches.

---

## Why not divide by the sum directly?

Suppose:

```python
[-1,2,3]
```

Direct normalization:

```python
[-1,2,3]/4
```

↓

```python
[-0.25,0.5,0.75]
```

Problems:

❌ Negative probabilities

❌ Doesn't sum to 1 properly in many cases

❌ Weak separation

---

Exponentiation fixes this:

```text
Always positive
```

because:

```python
eˣ > 0
```

for every x.

Therefore softmax always produces:

```text
Valid probabilities
```

---

## How does softmax turn scores into probabilities?

Example:

```python
[2,5,1]
```

Softmax:

```python
[0.05,0.91,0.04]
```

Now:

```text
0.05 + 0.91 + 0.04
=
1
```

Properties:

| Property | Result |
|----------|----------|
| All values ≥ 0 | Valid probabilities |
| Row sum = 1 | Distribution |
| Larger score → Larger probability | Ranking preserved |

---

## Similarity Score vs Attention Weight

| Similarity Score | Attention Weight |
|----------|----------|
| Before softmax | After softmax |
| Raw dot product | Probability |
| Can be negative | Always positive |
| Doesn't sum to 1 | Sums to 1 |
| Relevance signal | Attention allocation |

Example:

Before:

```python
[2,5,1]
```

After:

```python
[0.05,0.91,0.04]
```

---

## Mental Model

```text
Dot Product:
"How well do we match?"

Softmax:
"How much attention do I give?"
```

---

# Overall Self-Attention

## What information is stored in embeddings?

```python
x
```

Contains:

```text
Raw token representation
```

May encode:

```text
Meaning
Grammar
Position
Context
Etc.
```

---

## What information is stored in queries?

```python
q = xWq
```

Represents:

```text
What information am I looking for?
```

Not hardcoded.

Learned during training.

---

## What information is stored in keys?

```python
k = xWk
```

Represents:

```text
What information do I contain?
```

Again:

```text
Learned
Not programmed
```

---

## What information is stored in attention scores?

```python
wei = q @ k.T
```

Stores:

```text
How relevant is token j
to token i?
```

Shape:

```python
(T,T)
```

Each cell:

```python
wei[i,j]
```

is a similarity/relevance score.

---

## How does information flow?

```python
x
```

↓

```python
q = xWq
k = xWk
```

↓

```python
wei = q @ k.T
```

↓

```python
wei = softmax(wei)
```

↓

```python
attention weights
```

---

## What is being learned at each stage?

| Stage | Learns |
|---------|---------|
| Embeddings | Token representations |
| Wq | What features to search for |
| Wk | What features to expose |
| Dot Product | No learning (just computes similarity) |
| Softmax | No learning (normalization) |
| Attention Weights | Result of learned similarity |

---

## What is NOT learned?

These are fixed operations:

```python
@
```

```python
softmax()
```

```python
transpose()
```

No parameters.

No learning.

---

## What IS learned?

```python
Embedding Table
```

```python
Wq
```

```python
Wk
```

(and later Wv)

These contain trainable parameters.

---

## Complete Mental Model

```text
Token
   ↓
Embedding
   ↓
"What am I?"
   ↓
Projection (Wq/Wk)
   ↓
"What do I need?"
"What do I contain?"
   ↓
Dot Product
   ↓
How well do we match?
   ↓
Softmax
   ↓
How much attention should I give?
```

---

## The Entire Self-Attention Story in One Diagram

```text
x
│
├── Wq ──► q
│
└── Wk ──► k

q @ k.T
     ↓
Similarity Scores
     ↓
Softmax
     ↓
Attention Weights
     ↓
Focus on relevant tokens
```

---

## Quick Reference

| Object | Meaning |
|----------|----------|
| `x` | Token embedding |
| `q` | What I'm looking for |
| `k` | What I contain |
| `q @ k.T` | Relevance score |
| Softmax | Convert scores to probabilities |
| Attention Weight | Fraction of attention assigned |
| Learned Parameters | Embeddings, Wq, Wk |
| Fixed Operations | MatMul, Softmax, Transpose |

### One-Line Summary

```text
Embeddings store information.

Queries decide what to search for.

Keys advertise what information exists.

Dot products measure relevance.

Softmax converts relevance into attention.
```

# Deep Questions

## What is a learned representation space?

A representation space is simply:

```text
The space where vectors live.
```

Example:

```python
x = [1,2,3]
```

lives in:

```text
Embedding Space (R³)
```

After:

```python
q = xWq
```

```text
(1×3)
@
(3×2)
=
(1×2)
```

you get:

```python
q = [4,5]
```

which now lives in:

```text
Query Space (R²)
```

Different coordinates.

Different meaning.

---

## Why is it called a "learned" representation space?

Because:

```python
Wq
```

and

```python
Wk
```

are learned through backpropagation.

Initially:

```text
Random Space
```

After training:

```text
Useful Space
```

The model learns:

```text
How should tokens be represented
so similarity becomes useful?
```

---

## Why do learned projections help similarity measurement?

Suppose raw embedding contains:

```text
[
 tense,
 position,
 gender,
 subject,
 color,
 ...
]
```

Not every feature matters for attention.

Maybe for a particular task:

```text
subject
position
```

matter a lot.

```text
color
```

doesn't matter.

---

Without projections:

```python
x @ x.T
```

all features participate equally.

---

With projections:

```python
q = xWq
k = xWk
```

training can learn:

```text
Increase useful signals

Suppress irrelevant signals
```

before similarity is computed.

---

## Example

Raw embedding:

```python
[2,10,1]
```

Suppose:

```text
Feature 1 = important
Feature 2 = important
Feature 3 = noise
```

Learned projection:

```python
W =
[
 [5],
 [4],
 [0.1]
]
```

```text
(1×3)
@
(3×1)
=
(1×1)
```

Result:

```python
2×5 + 10×4 + 1×0.1

=
50.1
```

Feature 3 barely matters anymore.

The projection has learned:

```text
What deserves attention.
```

---

# What does

```text
"Attention is similarity in a learned space"
```

actually mean mathematically?

Without projections:

```python
x @ x.T
```

```text
(T×C)
@
(C×T)
=
(T×T)
```

Each score:

```python
x_i · x_j
```

Meaning:

```text
Similarity in embedding space
```

---

With projections:

```python
q = xWq
k = xWk
```

Then:

```python
q @ k.T
```

```text
(T×D)
@
(D×T)
=
(T×T)
```

Each score becomes:

```python
(x_iWq) · (x_jWk)
```

This is the important equation.

---

## Compare

Raw similarity:

```python
x_i · x_j
```

Learned similarity:

```python
(x_iWq) · (x_jWk)
```

Notice:

```text
Similarity is no longer measured
between raw embeddings.
```

Instead:

```text
Similarity is measured
after both embeddings have been transformed.
```

---

## What is the dot product measuring now?

Not:

```text
Are these embeddings similar?
```

But:

```text
Are these embeddings similar
according to features the model learned matter?
```

Huge difference.

---

## The Deepest Interpretation

Suppose training discovers:

```text
Subject information
```

is important.

Then:

```python
Wq
```

may learn to extract:

```text
"What subject am I looking for?"
```

while:

```python
Wk
```

learns:

```text
"What subject information do I contain?"
```

Now:

```python
(x_iWq) · (x_jWk)
```

becomes:

```text
How well does token j
provide the information token i needs?
```

---

## Full Mathematical Pipeline

```python
x
```

↓

```python
q = xWq
k = xWk
```

↓

```python
wei = q @ k.T
```

↓

```python
wei[i,j]
=
(x_iWq)
·
(x_jWk)
```

↓

```python
softmax(wei)
```

↓

```text
Attention Weights
```

---

## Mental Model

```text
Embedding Space

"Who am I?"

        ↓

Query Space

"What am I looking for?"

        ↓

Key Space

"What information do I contain?"

        ↓

Dot Product

"Do we match?"
```

---

## One-Line Summary

```text
Attention is not measuring similarity between tokens.

Attention is measuring similarity between learned representations of tokens.
```

Or mathematically:

```python
x_i · x_j
```

↓

```python
(x_iWq) · (x_jWk)
```

```text
Raw Similarity
        ↓
Learned Similarity
```